# PagedCore two-token layer trace on T4

Enable **Internet** and **T4 GPU**, then **Run All**. Exact working source and
frozen setup are embedded. No Git push or Input dataset is required.
Set `PAGEDCORE_EVIDENCE_WORKDIR` to override `/kaggle/working`.

This traces the standalone one-token prompt plus one decode step through all
22 layers. It records hidden input, normalization, projected Q/K/V, rotated Q/K
(V stays unrotated), actual pool K/V reads including both token slots, attention,
output projection, both residuals, post-attention normalization and MLP output.
Each comparison reports exact value equality, maximum error, tolerance ratio,
first differing coordinate, shapes, original strides, offset, dtype and storage
size. Layers are numbered 0 through 21. Layout differences are reported separately
from value differences. Raw tensors and observed FP32 attention operation inputs
and outputs are saved for every layer, including differences within tolerance.

An uninstrumented baseline must match the traced final logits exactly. The first
exact mismatch is selected in layer/operation order; attention substeps compare
the actual QK product, scaled scores and FP32 output before its FP16 cast.
If the first difference is the QK product, it replays the recorded operands with
their original shapes/strides/offsets, compares packed operands, and tests dense
geometry using the same paged operand values. These controls run outside serving.
K/V history and the appended slot are measured separately. An incomplete K/V
trace is marked not_measured or partial with counts and a nullable maximum;
the initialized zero in an older failed acceptance report is not a measurement.

Download the final ZIP even on failure. Inspect `evidence/layer-trace/trace.json`
and `raw/layer-00.pt` through `layer-21.pt`. `first_difference` identifies the
earliest observed value difference; `complete` and `instrumentation_controls`
must pass before interpreting it. All arithmetic, tolerances, ownership checks
and GPU acceptance tests remain unchanged. This is diagnostic evidence only.


In [ ]:
import base64
import hashlib
import io
import json
import os
import subprocess
import sys
import tempfile
import traceback
import zipfile
from pathlib import Path

from IPython.display import FileLink, display

PAYLOAD = "..."
SNAPSHOT_SHA256 = "5e54d25d6ab5419b3ddc19c9cf0b3720335ad62746a130112e0e94d6ea39daec"


output_override = os.environ.get("PAGEDCORE_EVIDENCE_WORKDIR")
working = (
    Path(output_override).expanduser().resolve()
    if output_override
    else Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path.cwd()
)
working.mkdir(parents=True, exist_ok=True)
run_root = Path(tempfile.mkdtemp(prefix="pagedcore-layer-trace-", dir=working))
source = run_root / "source"
evidence = run_root / "evidence"
evidence.mkdir()
payload = base64.b64decode(PAYLOAD)
assert hashlib.sha256(payload).hexdigest() == SNAPSHOT_SHA256
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(source)
identity = json.loads((source / "source-identity.json").read_text())
environment = dict(
    os.environ,
    CUDA_VISIBLE_DEVICES="0",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
    OMP_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    TOKENIZERS_PARALLELISM="false",
)
commands = []


def run(name, command, *, required=False):
    record = {"name": name, "argv": list(map(str, command)), "cwd": str(source)}
    commands.append(record)
    print(name, flush=True)
    with (evidence / (name + ".log")).open("w") as log:
        process = subprocess.Popen(
            command, cwd=source, env=environment, stdout=log, stderr=subprocess.STDOUT
        )
        try:
            while process.poll() is None:
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    print(name + " is running; progress is saved in " + str(log.name), flush=True)
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            record["interrupted"] = True
            raise
    record["returncode"] = process.returncode
    (evidence / "commands.json").write_text(json.dumps(commands, indent=2))
    print(name, "exit", process.returncode, flush=True)
    if required and process.returncode:
        raise RuntimeError(name + " failed; inspect its retained log")


def verify_source():
    checks = {
        name: hashlib.sha256((source / name).read_bytes()).hexdigest()
        for name in identity["sha256"]
    }
    assert checks == identity["sha256"], "source changed after notebook generation"
    return checks


try:
    verify_source()
    run("setup-uv", [sys.executable, "-m", "pip", "install", "uv==0.11.27"], required=True)
    run(
        "setup-frozen",
        [
            sys.executable,
            "-m",
            "uv",
            "sync",
            "--frozen",
            "--extra",
            "cuda",
            "--extra",
            "dev",
            "--python",
            (source / ".python-version").read_text().strip(),
        ],
        required=True,
    )
    python = source / ".venv/bin/python"
    run(
        "resolved-dependencies",
        [sys.executable, "-m", "uv", "pip", "freeze", "--python", str(python)],
    )
    run("gpu-inventory", ["nvidia-smi", "-q"])

    target_code = "import json,sys,torch; from pathlib import Path; from pagedcore.cli import collect_environment_diagnostics; assert torch.cuda.is_available() and torch.cuda.device_count()==1; assert 'T4' in torch.cuda.get_device_name(0); assert torch.__version__=='2.7.1+cu126'; p=torch.cuda.get_device_properties(0); Path(sys.argv[1]).write_text(json.dumps(dict(torch=torch.__version__,name=p.name,uuid=str(p.uuid),capability=[p.major,p.minor],visible_devices=torch.cuda.device_count(),total_memory=p.total_memory,environment=collect_environment_diagnostics()),indent=2))"
    run("target", [str(python), "-c", target_code, str(evidence / "target.json")], required=True)
    run(
        "checkpoint-verify",
        [
            str(python),
            "-m",
            "pagedcore.cli",
            "verify",
            "--manifest-path",
            str(evidence / "environment-manifest.json"),
        ],
        required=True,
    )
    run(
        "layer-trace",
        [
            str(python),
            "-m",
            "scripts.trace_two_token_decode",
            "--output-dir",
            str(evidence / "layer-trace"),
        ],
        required=True,
    )
    (evidence / "source-after.json").write_text(json.dumps(verify_source(), indent=2))
except BaseException:
    (evidence / "notebook-failure.txt").write_text(traceback.format_exc())
    print("Run incomplete; retained the failure and partial evidence.", flush=True)
finally:
    (evidence / "commands.json").write_text(json.dumps(commands, indent=2))
    (evidence / "notebook-run.json").write_text(
        json.dumps(
            {
                "snapshot_sha256": SNAPSHOT_SHA256,
                "base_revision": identity["base_revision"],
                "scope": "Two-token diagnostic, not decoder release acceptance",
                "environment_overrides": {
                    key: environment[key]
                    for key in (
                        "CUDA_VISIBLE_DEVICES",
                        "CUBLAS_WORKSPACE_CONFIG",
                        "OMP_NUM_THREADS",
                        "MKL_NUM_THREADS",
                    )
                },
            },
            indent=2,
        )
    )
    files = {
        path.relative_to(evidence).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
        for path in evidence.rglob("*")
        if path.is_file()
    }
    (evidence / "checksums.json").write_text(json.dumps(files, indent=2))
    archive_path = working / (run_root.name + ".zip")
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("source-snapshot.zip", payload)
        for path in evidence.rglob("*"):
            if path.is_file():
                archive.write(path, "evidence/" + path.relative_to(evidence).as_posix())
    print("DOWNLOAD", archive_path, "bytes", archive_path.stat().st_size, flush=True)
    display(FileLink(str(archive_path)))